# Worm demo: what DiverScan found

This notebook shows the results of DiverScan on a ready-made analysis, so you can see what the tool produces before running it on your own data.

The recordings are trajectories of the nematode *C. elegans* under two conditions, `preexp` (pre-exposed) and `naive`. A model with five attention branches was trained to tell the two apart. The cells below show the same results as notebook 05 of DiverScan: how well the two conditions were told apart, where on the trajectories the model looked, and which movement features go with those places.

**How to run.** Sign in to a Google account and select `Runtime` -> `Run all`. Nothing is saved to your Google Drive: the notebook downloads the demo files it needs into the Colab session's temporary disk, and they are gone when the session ends.


In [ ]:
#@title ⬇️ Download the demo files into this session
#@markdown Nothing is written to your Google Drive. The files are downloaded into this Colab session's temporary disk and are deleted when the session ends.

import subprocess
import sys
import zipfile
from pathlib import Path

# DiverScan-release.zip in the shared Google Drive folder; only the parts of
# the archive listed in wanted_prefixes are extracted.
archive_id = "1T2c3p_3-ofSY8zf8pNpes1oac6PhOiwR"
wanted_prefixes = ["projects/worm_demo/"]

work_folder = Path("/content") if Path("/content").is_dir() else Path.cwd()
archive_path = work_folder / "DiverScan-release.zip"
project_folder = work_folder / "DiverScan-release" / "projects" / "worm_demo"

if not project_folder.is_dir():
    if not archive_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        gdown.download(id=archive_id, output=str(archive_path), quiet=False)
    with zipfile.ZipFile(archive_path) as archive:
        wanted = [m for m in archive.namelist() if m.startswith(tuple(wanted_prefixes))]
        archive.extractall(work_folder / "DiverScan-release", members=wanted)
    archive_path.unlink()  # the archive is no longer needed
print("demo files:", project_folder)


# ⚙️ Results

Scroll down to the results. The numbered cells can be run again on their own after changing their settings.


In [ ]:
#@title ⚙️ Setting
#@markdown Loads the demo project. There is nothing to fill in.

import os
import sys
from pathlib import Path
import pickle
import gzip
import random
import numpy as np
import pandas as pd

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Running in Google Colab environment.")
except ImportError:
    # Not running in Google Colab environment
    pass

# For reproductivity
seed = 0
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)

# The demo project, downloaded into this session by the cell above.
project_folder = Path(project_folder)

dataset_folder = project_folder / "dataset"
results_folder = project_folder / "results"
utils_folder = str(project_folder / "utils_demo")
epoch_visualization_folder = results_folder  # the cells below read their files from here

# How the demo model was set up: one animal per recording, classification,
# nodes not shuffled. The number of attention branches is read from the results.
model_type = "single"
task_type = "classification"
node_shuffle = False

# The code that draws the plots travels with the demo project.
if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import postprocessing as postprocessing_module


def read_settings_file(path):
    values = {}
    with open(path, "r") as f:
        for line in f:
            key, separator, value = line.rstrip("\n").partition(": ")
            if separator:
                values[key] = value
    return values


# The dataset: the unstandardized features of the test set, for the plots.
os.chdir(dataset_folder)
print(dataset_folder)
loaded_values = read_settings_file("settings_of_preprocessing.txt")
if "moving_window_size" in loaded_values:
    print("moving window size:", loaded_values["moving_window_size"], "frames")
with gzip.open("dataset_unstandardized_for_post_analysis.pkl", "rb") as file:
    post_analysis_dictionary = pickle.load(file)

# The results: predictions and attention of the model on the test set.
os.chdir(results_folder)
print(results_folder)
print("best epoch:", read_settings_file("best_epoch.txt").get("epoch", open("best_epoch.txt").readline().strip()))

with gzip.open("prediction_attention_namelist.pkl", "rb") as file:
    loaded_data = pickle.load(file)
attentions = loaded_data["attentions"]
class_name_list = loaded_data["class_name_list"]
feature_name_list = loaded_data["feature_name_list"]
edge_data_name_list = loaded_data["edge_data_name_list"]
number_of_attention_branches = attentions.shape[1]
if model_type == "multi":
    pred_masks = loaded_data["pred_masks"]

with gzip.open("dictionary_for_visualization.pkl", "rb") as file:
    dictionary = pickle.load(file)

print("conditions:", class_name_list)
print("attention branches:", number_of_attention_branches)


In [ ]:
#@title 1️⃣ Model performance
#@markdown You can check the performance of the attention branches.
from IPython.display import display


if task_type == "classification":

    print("⭐️ Macro averaged accuracy (test set) ⭐️")
    df = pd.read_excel("macro_averaged_accuracy.xlsx")
    df.rename(
        columns=lambda x: '' if isinstance(x, str) and x.startswith('Unnamed') else x,
        inplace=True
    )
    display(df)
    print("\n\n⭐️ Classification report (test set) ⭐️\n")

    classification_report_dictionary = pd.read_excel("classification_report.xlsx", sheet_name=None)
    postprocessing_module.show_classification_report(
        classification_report_dictionary,
        number_of_attention_branches,
    )
    print("\nFor details, see the document of sklearn.metrics.classification_report. https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html")

In [ ]:
#@title 2️⃣ PCA of attention timeseries
#@markdown You can check which attention branches are distinctive and which are similar to each other.

flatten_attentions = attentions.transpose(1, 0, 2, 3).reshape(attentions.shape[1], -1)  # (number_of_attention_branches, files*N*T)

postprocessing_module.create_interactive_pca_plot(
    flatten_attentions,
    "Default",
    scale=20,
)

In [ ]:
#@title 3️⃣ Visualization of attention

default_filename_list = dictionary[class_name_list[0]].keys()
_, number_of_nodes, _ = dictionary[class_name_list[0]][list(default_filename_list)[0]]["attentions"].shape

postprocessing_module.make_plots_and_gui(
    dictionary,
    model_type,
    class_name_list,
    default_filename_list,
    number_of_attention_branches,
    feature_name_list,
    number_of_nodes,
    dpi=100,
    xlim=None,  # e.g., (0, 100) or None
    ylim=None,  # e.g., (0, 100) or None
)

In [ ]:
#@title 4️⃣ Interactive plot
#@markdown Select the left or right plot of 3️⃣ which you want to create a interactive one.

postprocessing_module.make_interactive_plot_and_gui(
    dictionary,
    model_type,
    sampling_interval=10,
)

In [ ]:
#@title 5️⃣ Correlation and Hellinger distance
#@markdown You can see potential features that contribute to attention maps.
#@markdown - Select what to show from the dropdown; the content updates right away.
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown - Hellinger distance is a measure of the similarity between two distributions.
#@markdown - When the Hellinger distance is 0, the two distributions are identical, and when it is 1, they are completely different.
#@markdown
#@markdown **[Multi mode] indicators** (available when model_type == "multi"):
#@markdown - `differences of basic features between nodes`: sum of absolute differences between each node's feature and those of the other nodes. Higher when the node behaves differently from the rest of the group.
#@markdown - `edge features`: mean / variance over all edges (broadcast to every node as the same value). Reflects the overall cohesion and variability of the group.
#@markdown - `incident edge features per node`: per-node sum / min / max / variance of the edges incident to the node. Distinguishes isolated individuals from central individuals on a node-by-node basis.
#@markdown - `MST features`: degree / is_leaf / incident sum / incident max on the Minimum Spanning Tree built from the distance edge. Captures structural roles such as leaf, hub, or bridge.
#@markdown - `geometric features`: distance to the group centroid / local density / head-direction alignment computed from (x, y) coordinates (only when coordinates are available).

show_options = ["Correlation (basic features)", "Correlation in highlighted segments (basic features)", "Hellinger distance in highlighted segments (basic features)", "[Multi mode] Correlation (differences of basic features between nodes)", "[Multi mode] Correlation in highlighted segments (differences of basic features between nodes)", "[Multi mode] Hellinger distance in highlighted segments (differences of basic features between nodes)", "[Multi mode] Correlation (edge features)", "[Multi mode] Correlation in highlighted segments (edge features)", "[Multi mode] Hellinger distance in highlighted segments (edge features)", "[Multi mode] Correlation (incident edge features per node)", "[Multi mode] Correlation in highlighted segments (incident edge features per node)", "[Multi mode] Hellinger distance in highlighted segments (incident edge features per node)", "[Multi mode] Correlation (MST features)", "[Multi mode] Correlation in highlighted segments (MST features)", "[Multi mode] Hellinger distance in highlighted segments (MST features)", "[Multi mode] Correlation (geometric features)", "[Multi mode] Correlation in highlighted segments (geometric features)", "[Multi mode] Hellinger distance in highlighted segments (geometric features)"]


postprocessing_module.show_selected_result_gui(
    show_options,
    "Correlation (basic features)",
    class_name_list,
    number_of_attention_branches=number_of_attention_branches,
    model_type=model_type,
)


In [ ]:
#@title 6️⃣ Histogram of features in the highlighted segments
#@markdown You can see distribution differences of feature values of two conditions in the highlighted segments.
#@markdown - Select what to show from the dropdown; the content updates right away.
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown
#@markdown **[Multi mode] indicators** (available when model_type == "multi"):
#@markdown - `differences of basic features between nodes`: sum of absolute differences between each node's feature and those of the other nodes. Higher when the node behaves differently from the rest of the group.
#@markdown - `edge features`: mean / variance over all edges (broadcast to every node as the same value). Reflects the overall cohesion and variability of the group.
#@markdown - `incident edge features per node`: per-node sum / min / max / variance of the edges incident to the node. Distinguishes isolated individuals from central individuals on a node-by-node basis.
#@markdown - `MST features`: degree / is_leaf / incident sum / incident max on the Minimum Spanning Tree built from the distance edge. Captures structural roles such as leaf, hub, or bridge.
#@markdown - `geometric features`: distance to the group centroid / local density / head-direction alignment computed from (x, y) coordinates (only when coordinates are available).

show_options = ["Histogram in highlighted segments (basic features)", "[Multi mode] Histogram in highlighted segments (differences of basic features between nodes)", "[Multi mode] Histogram in highlighted segments (edge features)", "[Multi mode] Histogram in highlighted segments (incident edge features per node)", "[Multi mode] Histogram in highlighted segments (MST features)", "[Multi mode] Histogram in highlighted segments (geometric features)"]


postprocessing_module.show_selected_result_gui(
    show_options,
    "Histogram in highlighted segments (basic features)",
    class_name_list,
    feature_name_list=feature_name_list,
    dpi=100,
    model_type=model_type,
)


In [ ]:
#@title Model prediction of all files in the test set
#@markdown "True" indicates that the model prediction is correct, while "False" indicates it is incorrect. For more details, go to the `epochXX_for_visualization` folder.

prediction_true_false = pd.read_excel("prediction_true_false.xlsx", sheet_name=None)

postprocessing_module.show_table_and_gui(
    prediction_true_false,
    list(prediction_true_false.keys()),
    dropdown_name="condition",
)

In [ ]:
#@title Make summary tables (SummaryTable.xlsx)

import subprocess
from pathlib import Path

SCRIPT_PATH  = Path(utils_folder) / "make_summary_table.py"
INPUT_DIR    = Path(epoch_visualization_folder)
OUTPUT_XLSX  = INPUT_DIR / "SummaryTable.xlsx"

CLASS_NAMES = class_name_list
NUM_ATTENTIONS = number_of_attention_branches
MODEL_TYPE = model_type
# --------------------------------------

cmd = [
    "python",
    str(SCRIPT_PATH),
    str(INPUT_DIR),
    "--class-names", str(CLASS_NAMES),
    "--num-attentions", str(NUM_ATTENTIONS),
    "--model-type", str(MODEL_TYPE),
    "-o", str(OUTPUT_XLSX),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Saved:", OUTPUT_XLSX.resolve())
